# Chord Transformer — Training on Colab

Train the chord language model on a T4 GPU using the Chordonomicon dataset (666K songs).

**Setup:** Runtime → Change runtime type → T4 GPU

## 1. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/shawnjiang1019/chord-transformer.git
%cd chord-transformer
!pip install datasets pyyaml

## 2. Mount Google Drive for checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/chord_checkpoints'
!mkdir -p {CHECKPOINT_DIR}

## 3. Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 4. Load tokenizer and dataset

In [ ]:
import sys
sys.path.insert(0, '/content/chord-transformer')

from src.data.tokenizer import ChordTokenizer
from src.data.dataset import ChordDataset

tokenizer = ChordTokenizer()
print(f'Vocab size: {tokenizer.vocab_size}')

# Set max_songs=1000 for a quick test, None for full dataset
MAX_SONGS = None
MAX_SEQ_LEN = 512

print('Loading dataset from HuggingFace (streaming)...')
full_ds = ChordDataset(tokenizer, max_seq_len=MAX_SEQ_LEN, max_songs=MAX_SONGS)
print(f'Loaded {len(full_ds)} songs')

## 5. Train/val split and dataloaders

In [ ]:
from torch.utils.data import DataLoader, random_split

BATCH_SIZE = 64
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1

n_total = len(full_ds)
n_train = int(n_total * TRAIN_SPLIT)
n_val = int(n_total * VAL_SPLIT)
n_test = n_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42),
)
print(f'Split: {n_train} train / {n_val} val / {n_test} test')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

## 6. Create model

In [ ]:
from src.model.transformer import ChordTransformer

model = ChordTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=256,
    n_heads=4,
    n_layers=4,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} parameters')

## 7. Train

In [ ]:
from src.model.train import train

model = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=1e-4,
    warmup_steps=1000,
    max_grad_norm=1.0,
    device='cuda',
    checkpoint_dir=CHECKPOINT_DIR,
)

## 8. Test generation

In [ ]:
model.eval()
model = model.to('cuda')

# Prompt: verse section starting with C major
prompt = tokenizer.encode('<verse> C')
# Remove [EOS] from prompt since we want the model to continue
prompt = prompt[:-1]
prompt_ids = torch.tensor([prompt], device='cuda')

print(f'Prompt: {tokenizer.decode(prompt)}')
print()

# Generate with different temperatures
for temp in [0.7, 1.0, 1.3]:
    output = model.generate(prompt_ids, max_new_tokens=30, temperature=temp)
    chords = tokenizer.decode(output[0].tolist())
    print(f'temp={temp}: {chords}')

## 9. Download best checkpoint

In [ ]:
# If you didn't mount Drive, download the checkpoint directly
# from google.colab import files
# files.download(f'{CHECKPOINT_DIR}/best_model.pt')

print(f'Checkpoints saved to: {CHECKPOINT_DIR}')
!ls -lh {CHECKPOINT_DIR}